# Modèle final : prédiction de la réadmission < 30 jours

**Objectif** : entraîner le modèle retenu (forêt aléatoire, class_weight="balanced"),
définir des niveaux de risque, reconstituer le clustering et sauvegarder
tous les éléments nécessaires à l'application.

**Choix issus du notebook 1** : split par patient 65/35, forêt aléatoire
(meilleures AUC), class_weight="balanced" (meilleur que SMOTE), k=5 pour le clustering.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             recall_score, precision_score, f1_score)

# keep_default_na=False : évite que pandas transforme une modalité texte en NaN
df = pd.read_csv("../data/processed/diabetes_clean_clusters.csv", keep_default_na=False)
print(df.shape)
print("Manquants :", df.isna().sum().sum())
df.head(3)

(99340, 39)
Manquants : 0


,encounter_id,patient_nbr,race,gender,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,...,readmit_30,discharge_group,admission_type,admission_source,age_num,n_meds,n_med_changes,total_visits,cluster_k4,cluster_k5
0,2278392,8222157,Caucasian,Female,1,Other,41,0,1,0,...,0,Other,Unknown,Referral,5,0,0,0,2,2
1,149190,55629189,Caucasian,Female,3,Unknown,59,0,18,0,...,0,Home,Emergency,Emergency_room,15,1,1,0,3,4
2,64410,86047875,AfricanAmerican,Female,2,Unknown,11,5,13,2,...,0,Home,Emergency,Emergency_room,25,1,0,3,1,1


In [2]:
drop_cols = ["encounter_id", "patient_nbr", "readmitted", "readmit_30",
             "cluster_k4", "cluster_k5"]
X = df.drop(columns=drop_cols)
y = df["readmit_30"]
groups = df["patient_nbr"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.35, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train :", X_train.shape, "| Test :", X_test.shape)
print("Positifs train :", round(y_train.mean(), 4), "| test :", round(y_test.mean(), 4))
print("Patients communs :", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))

Train : (64652, 33) | Test : (34688, 33)
Positifs train : 0.1136 | test : 0.1144
Patients communs : 0


In [3]:
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()
print(len(cat_cols), "catégorielles |", len(num_cols), "numériques")

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=30,
                                   class_weight="balanced", n_jobs=-1, random_state=42)),
])
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
pred = model.predict(X_test)
print("AUC-ROC :", round(roc_auc_score(y_test, proba), 3))
print("AUC-PR  :", round(average_precision_score(y_test, proba), 3))
print("Recall  :", round(recall_score(y_test, pred), 3))
print("F1      :", round(f1_score(y_test, pred), 3))

17 catégorielles | 16 numériques
AUC-ROC : 0.663
AUC-PR  : 0.209
Recall  : 0.576
F1      : 0.274


## Niveaux de risque
Avec class_weight="balanced", les probabilités sont gonflées : on les convertit
en niveaux de risque via des quantiles des probabilités du jeu de test
(Élevé = 10 % les plus hauts, Moyen = les 20 % suivants, Faible = 70 %).
Limite : seuils fixés sur le test (pas de jeu de validation distinct).

In [4]:
t_high = float(np.quantile(proba, 0.90))
t_mid = float(np.quantile(proba, 0.70))
print(f"Seuil Moyen >= {t_mid:.3f} | Seuil Élevé >= {t_high:.3f}")

def risk_level(p):
    return "Élevé" if p >= t_high else ("Moyen" if p >= t_mid else "Faible")

res = pd.DataFrame({"proba": proba, "y": y_test.values})
res["niveau"] = res["proba"].apply(risk_level)
base = res["y"].mean()

tab = (res.groupby("niveau")
       .agg(patients=("y", "size"), positifs=("y", "sum"), taux=("y", "mean"))
       .reindex(["Faible", "Moyen", "Élevé"]))
tab["part"] = (tab["patients"] / len(res)).round(3)
tab["lift"] = (tab["taux"] / base).round(2)
tab["taux"] = tab["taux"].round(3)
display(tab)
print("Taux global :", round(base, 4))

Seuil Moyen >= 0.523 | Seuil Élevé >= 0.610


,patients,positifs,taux,part,lift
niveau,,,,,
Faible,24281,1995,0.082,0.7,0.72
Moyen,6938,1122,0.162,0.2,1.41
Élevé,3469,852,0.246,0.1,2.15


Taux global : 0.1144


In [5]:
clust_cols = ["age_num", "time_in_hospital", "num_lab_procedures", "num_procedures",
              "num_medications", "number_diagnoses", "number_outpatient",
              "number_emergency", "number_inpatient", "n_meds", "n_med_changes"]

def clustering_features(d):
    Z = d[clust_cols].copy()
    Z["change"] = (d["change"] == "Ch").astype(int)
    Z["diabetesMed"] = (d["diabetesMed"] == "Yes").astype(int)
    return Z

scaler_clu = StandardScaler().fit(clustering_features(df))
kmeans = KMeans(n_clusters=5, n_init=10, random_state=42).fit(
    scaler_clu.transform(clustering_features(df)))

accord = (kmeans.labels_ == df["cluster_k5"].values).mean()
print(f"Accord avec les clusters du CSV : {accord:.4f}")

Accord avec les clusters du CSV : 1.0000


In [6]:
cluster_names = {
    0: "Patients récurrents à forte utilisation des soins",
    1: "Séjours courts, monothérapie stable",
    2: "Séjours courts sans traitement antidiabétique",
    3: "Séjours longs et complexes",
    4: "Séjours courts, bithérapie avec ajustements",
}
cluster_rate = df.groupby("cluster_k5")["readmit_30"].mean().round(3).to_dict()

config = {
    "thresholds": {"mid": t_mid, "high": t_high},
    "risk_rates": {k: float(v) for k, v in tab["taux"].items()},
    "global_rate": float(base),
    "feature_order": X.columns.tolist(),
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "cat_options": {c: sorted(X[c].unique().tolist()) for c in cat_cols},
    "num_ranges": {c: [float(X[c].min()), float(X[c].max()), float(X[c].median())]
                   for c in num_cols},
    "cluster_names": {str(k): v for k, v in cluster_names.items()},
    "cluster_rates": {str(k): float(v) for k, v in cluster_rate.items()},
}
print(len(config["feature_order"]), "variables d'entrée")
print(config["cat_options"]["discharge_group"])

33 variables d'entrée
['Home', 'Home_care', 'Other', 'Transfer']


In [7]:
joblib.dump(model, "../models/rf_model.joblib")
joblib.dump(scaler_clu, "../models/scaler_clustering.joblib")
joblib.dump(kmeans, "../models/kmeans_k5.joblib")
with open("../models/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print("Sauvegardé dans models/")

Sauvegardé dans models/


In [8]:
def predict_patient(row: dict):
    d = pd.DataFrame([row])
    p = float(model.predict_proba(d[config["feature_order"]])[:, 1][0])
    c = int(kmeans.predict(scaler_clu.transform(clustering_features(d)))[0])
    return {"proba": round(p, 3), "niveau": risk_level(p),
            "cluster": c, "profil": cluster_names[c]}

# 3 patients du jeu de test : le plus à risque, un médian, le moins à risque
ordre = np.argsort(proba)
for i in [ordre[-1], ordre[len(ordre)//2], ordre[0]]:
    ligne = X_test.iloc[i].to_dict()
    print(predict_patient(ligne), "| proba du pipeline :", round(proba[i], 3),
          "| réadmis réellement :", int(y_test.iloc[i]))

{'proba': 0.815, 'niveau': 'Élevé', 'cluster': 0, 'profil': 'Patients récurrents à forte utilisation des soins'} | proba du pipeline : 0.815 | réadmis réellement : 0
{'proba': 0.455, 'niveau': 'Faible', 'cluster': 1, 'profil': 'Séjours courts, monothérapie stable'} | proba du pipeline : 0.455 | réadmis réellement : 0
{'proba': 0.187, 'niveau': 'Faible', 'cluster': 1, 'profil': 'Séjours courts, monothérapie stable'} | proba du pipeline : 0.187 | réadmis réellement : 0
